# Qwen2.5-Coder-0.5B — Python FIM LoRA Fine-tuning
**Kaggle notebook — thin controller only. All logic lives in the GitHub repo.**

This notebook:
1. Clones the repo
2. Installs dependencies
3. Sets the HF token from Kaggle Secrets
4. Copies the FIM dataset from the attached Kaggle Dataset
5. Runs training (`src/training/train_lora.py`)
6. Runs evaluation (`src/evaluate.py`)

**Required setup before running:**
- Accelerator: GPU T4 x2
- Internet: ON
- Add Kaggle Dataset: `fim-python-dataset`
- Add Kaggle Secret: `HF_TOKEN` (your Hugging Face write token)

In [ ]:
# ── Cell 1: Clone repository at the requested Git state ────────────────────
import os
import shutil
import subprocess

GITHUB_REPO = "https://github.com/Rudra-G-23/qwen2.5-coder-0.5b-python-fim.git"

# Set these as needed
BRANCH = "feat/stage-one"  # None -> main
COMMIT = None  # None -> latest commit of branch
ATTEMPT = 1  # increment each time you re-run this config -> distinct W&B run ID (...-a1, -a2, ...)

# Leave "" for normal behavior (auto-built run_id; if this exact config was
# already training, it resumes automatically). Paste an exact run_id here
# ONLY to force-resume a specific crashed/interrupted run instead -- e.g.
# copy it straight from that run's title in the W&B UI:
#   "qwen05b-lora-r16-e1-dsdistributed-20260821-a42"
# This fetches that run's HF checkpoint (mid-training weights) and continues
# the SAME W&B run's log/graphs (status flips Crashed -> Running) instead of
# starting over or forking a new run. Errors loudly if that run_id has no
# matching HF checkpoint or W&B run -- so a typo fails fast instead of
# silently training from scratch.
RESUME_RUN_ID = ""

REPO_DIR = "/kaggle/working/repo"

# ── Defaults ────────────────────────────────────────────────────────────────
if BRANCH is None:
    BRANCH = "main"

# ── Clean previous clone ────────────────────────────────────────────────────
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

# ── Case 1: Exact commit requested ─────────────────────────────────────────
if COMMIT:
    print(f"Cloning repository...")
    print(f"Branch : {BRANCH}")
    print(f"Commit : {COMMIT}")

    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            "--single-branch",
            GITHUB_REPO,
            REPO_DIR,
        ],
        check=True,
    )

    os.chdir(REPO_DIR)

    subprocess.run(["git", "checkout", "--detach", COMMIT], check=True)

# ── Case 2: Branch requested, but no commit ────────────────────────────────
else:
    print(f"Cloning latest commit from branch: {BRANCH}")

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            BRANCH,
            "--single-branch",
            GITHUB_REPO,
            REPO_DIR,
        ],
        check=True,
    )

    os.chdir(REPO_DIR)

# ── Verify final state ─────────────────────────────────────────────────────
current_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip()

current_branch = subprocess.check_output(
    ["git", "branch", "--show-current"], text=True
).strip()

print("\n✓ Repository ready")
print(f"  Branch : {current_branch or '(detached HEAD)'}")
print(f"  Commit : {current_commit}")
print(f"  Attempt: {ATTEMPT}")
if RESUME_RUN_ID:
    print(f"  Resume : {RESUME_RUN_ID}  (explicit — will force-resume this run)")


In [ ]:
# ── GPU check — stop early on a CPU-only session ─────────────────────────────
# This notebook needs a GPU (LoRA training via unsloth). If Kaggle assigned a
# CPU-only session (accelerator not set to GPU, or GPU quota exhausted), stop
# right here instead of burning the session on installs/training that will
# crash or run unusably slowly on CPU.
import subprocess

try:
    subprocess.run(["nvidia-smi"], check=True, capture_output=True)
    print("\u2713 GPU detected \u2014 continuing.")
except (subprocess.CalledProcessError, FileNotFoundError):
    raise SystemExit(
        "\u2717 No GPU detected. This notebook requires a GPU \u2014 set the accelerator in "
        "Kaggle: Settings (right sidebar) \u2192 Accelerator \u2192 GPU T4 x2 (or similar), then "
        "Save & re-run. Stopping now to save your session quota."
    )


In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
        "peft>=0.20.0",
        "trl>=0.17.0",
        "transformers>=5.0.0",
        "datasets>=3.0.0",
        "pyyaml>=6.0",
        "huggingface_hub>=0.30.0",
        "seaborn>=0.13.0",
        "pandas>=2.0.0",
    ],
    check=True,
)

print("✓ Dependencies installed")


In [ ]:
# ── Cell 2: Authenticate to Hugging Face ──────────────────────────────────
# HF_TOKEN is stored as a Kaggle Secret — NEVER hardcode tokens.
# Add it: Kaggle account → Settings → Secrets → Add New Secret
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
print("✓ HF_TOKEN set from Kaggle Secrets")

In [ ]:
# ── Cell 3: Copy dataset from Kaggle Dataset input ──────────────────────────
# In the Kaggle sidebar: Add Data → Your Datasets → fim-python-dataset
import pathlib
import shutil

# Kaggle mounts dataset inputs at /kaggle/input/<dataset-slug>/
DATASET_INPUT = (
    "/kaggle/input/datasets/rudraprasadbhuyan/fim-python-dataset/data/fim_dataset.jsonl"
)
DATASET_DEST = f"{REPO_DIR}/data/fim_dataset.jsonl"

pathlib.Path(f"{REPO_DIR}/data").mkdir(parents=True, exist_ok=True)
shutil.copy(DATASET_INPUT, DATASET_DEST)

# Count examples
n = sum(1 for line in open(DATASET_DEST) if line.strip())
print(f"✓ Dataset copied: {n} FIM examples")


In [ ]:
# ── Cell 4: Run LoRA training ─────────────────────────────────────────────
import os
import sys

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from src.training.train_lora import train

# finish_wandb=False keeps this run open so Cell 5 logs eval plots/tables
# into the SAME W&B run instead of orphaning them into a separate one.
wandb_run = train(
    config_path="configs/training/lora.yaml",
    overrides={
        "wandb": {"attempt": ATTEMPT},
        "training": {"resume_run_id": RESUME_RUN_ID},
    },
    finish_wandb=False,
)

In [ ]:
# ── Cell 5: Evaluate — Base vs LoRA-FT ───────────────────────────────────
from src.training.evaluate import run_evaluation

run_evaluation(
    base_model_name="Qwen/Qwen2.5-Coder-0.5B",
    adapter_path="/kaggle/working/checkpoints/lora_adapter",
    test_data_path="data/fim_dataset.jsonl",
    output_dir="/kaggle/working/results",
    max_samples=200,
    trainer_log_path="/kaggle/working/checkpoints/trainer_state.json",
    wandb_run=wandb_run,
)

if wandb_run is not None:
    wandb_run.finish()
    print(f"View run + plots + hyperparameter table: {wandb_run.url}")

In [ ]:
# ── Cell 6: Display plots inline ─────────────────────────────────────────
from IPython.display import Image, display

for plot in [
    "/kaggle/working/results/plots/loss_curve.png",
    "/kaggle/working/results/plots/comparison.png",
]:
    if __import__("pathlib").Path(plot).exists():
        print(f"\n{plot}")
        display(Image(filename=plot))

## Resuming next week

```python
# Load your saved adapter from HF on top of a fresh base model:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base  = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-Coder-0.5B', device_map='auto')
model = PeftModel.from_pretrained(base, 'Rudra-G-23/qwen2.5-coder-0.5b-python-fim')
# Then continue training or just evaluate.
```

The base model never changes. Only your adapter accumulates updates.